# 03 — Search and Optimization

Soma includes a built-in hyperparameter optimization engine with three strategies:

- **Grid**: Exhaustive search over all parameter combinations
- **Random**: Uniform sampling from the search space
- **Bayesian**: TPE-based (Tree-Parzen Estimator) — learns from past trials

This notebook covers:
- Defining search spaces with `search()` descriptors
- Running a `Study` with different strategies
- Inspecting results and best trials

## 3.1 — Search descriptors

The `search()` function creates descriptors that define the search space
for a filter's parameters. They work as Python descriptors on the class.

In [ ]:
from soma import Filter, Pipeline, Study, search

class MyModel(Filter):
    # Float parameter with log scale (common for learning rates)
    lr: float = search(0.001, 0.1, scale="log")

    # Integer parameter
    n_layers: int = search(1, 5)

    # Categorical parameter
    activation: str = search(choices=["relu", "tanh", "sigmoid"])

    # Boolean auto-detected as categorical [True, False]
    use_bias: bool = search()

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

# Inspect the collected search space
for dim in MyModel._soma_search_space:
    print(dim)

## 3.2 — Your first Study (Grid Search)

A `Study` runs an **executor function** for each trial. The executor receives
a dict of parameter values and must return a dict of metric values.

In [ ]:
# Objective: find the value of x that maximizes f(x) = 1 - |x - 0.5| * 2
# The optimum is at x = 0.5, where f(0.5) = 1.0

def objective(params):
    x = params["x"]
    score = max(0.0, 1.0 - abs(x - 0.5) * 2)
    return {"score": score}

# Grid search: 11 evenly spaced points in [0, 1]
grid_study = Study(
    name="grid_example",
    search_space=[
        {"type": "float", "name": "x", "low": 0.0, "high": 1.0, "scale": "linear"},
    ],
    strategy="grid",
    n_trials=11,        # 11 points per dimension
    objectives=[("score", "maximize")],
)

grid_study.run(objective)

print(f"Trials run: {grid_study.n_trials}")
print(f"Progress:   {grid_study.progress:.0%}")

best = grid_study.best_trial
print(f"\nBest trial:")
print(f"  x     = {best['params']['x']:.3f}")
print(f"  score = {best['metrics']['score']:.3f}")

## 3.3 — Random Search

Random search samples uniformly from the search space. Often more efficient
than grid search for high-dimensional spaces because it doesn't waste trials
on unimportant dimensions.

In [ ]:
# 2D search: find (x, y) that minimizes (x - 0.3)^2 + (y - 0.7)^2
# Optimum at (0.3, 0.7)

def objective_2d(params):
    x = params["x"]
    y = params["y"]
    loss = (x - 0.3) ** 2 + (y - 0.7) ** 2
    return {"loss": loss}

random_study = Study(
    name="random_2d",
    search_space=[
        {"type": "float", "name": "x", "low": 0.0, "high": 1.0, "scale": "linear"},
        {"type": "float", "name": "y", "low": 0.0, "high": 1.0, "scale": "linear"},
    ],
    strategy="random",
    n_trials=50,
    objectives=[("loss", "minimize")],
    seed=42,
)

random_study.run(objective_2d)

best = random_study.best_trial
print(f"Best of {random_study.n_trials} random trials:")
print(f"  x    = {best['params']['x']:.3f}  (target: 0.300)")
print(f"  y    = {best['params']['y']:.3f}  (target: 0.700)")
print(f"  loss = {best['metrics']['loss']:.6f}")

## 3.4 — Bayesian Optimization (TPE)

Bayesian optimization uses the history of past trials to make smarter choices.
It splits trials into "good" (top 25%) and "bad", then samples near the good ones.

The first `n_startup` trials are random (exploration), then TPE kicks in.

In [ ]:
# Same 2D problem, but with Bayesian optimization
bayesian_study = Study(
    name="bayesian_2d",
    search_space=[
        {"type": "float", "name": "x", "low": 0.0, "high": 1.0, "scale": "linear"},
        {"type": "float", "name": "y", "low": 0.0, "high": 1.0, "scale": "linear"},
    ],
    strategy="bayesian",
    n_trials=50,
    objectives=[("loss", "minimize")],
    seed=42,
)

bayesian_study.run(objective_2d)

best_b = bayesian_study.best_trial
print(f"Bayesian best ({bayesian_study.n_trials} trials):")
print(f"  x    = {best_b['params']['x']:.3f}  (target: 0.300)")
print(f"  y    = {best_b['params']['y']:.3f}  (target: 0.700)")
print(f"  loss = {best_b['metrics']['loss']:.6f}")

print(f"\nComparison (same budget, 50 trials):")
print(f"  Random:   loss = {best['metrics']['loss']:.6f}")
print(f"  Bayesian: loss = {best_b['metrics']['loss']:.6f}")

## 3.5 — Mixed search spaces (float + categorical)

Studies can mix different dimension types. Grid search computes the cartesian
product of all dimensions.

In [ ]:
# Simulate choosing a model type and a learning rate
def mixed_objective(params):
    lr = params["lr"]
    model = params["model"]

    # "svm" works best at high lr, "tree" at low lr
    if model == "svm":
        score = 1.0 - abs(lr - 0.08) * 10
    elif model == "tree":
        score = 1.0 - abs(lr - 0.005) * 100
    else:
        score = 0.5
    return {"score": max(0.0, score)}

mixed_study = Study(
    name="mixed_search",
    search_space=[
        {"type": "float", "name": "lr", "low": 0.001, "high": 0.1, "scale": "log"},
        {"type": "categorical", "name": "model", "choices": ["svm", "tree", "linear"]},
    ],
    strategy="grid",
    n_trials=5,  # 5 points for lr × 3 choices = 15 total
    objectives=[("score", "maximize")],
)

mixed_study.run(mixed_objective)

best = mixed_study.best_trial
print(f"Total trials: {mixed_study.n_trials}")
print(f"Best: model={best['params']['model']}, lr={best['params']['lr']:.4f}")
print(f"Score: {best['metrics']['score']:.3f}")

## 3.6 — Using search descriptors with Filters

Instead of manually building the search space dict, you can define it
directly on the Filter class using `search()` descriptors.

In [ ]:
class RidgeRegressor(Filter):
    """A mock ridge regression with searchable hyperparameters."""
    alpha: float = search(0.001, 100.0, scale="log")
    normalize: bool = search()  # auto: [True, False]

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def fit(self, x, y=None):
        # Simulate learning coefficients
        return {"coef": [0.5] * len(x)}

    def forward(self, x, state):
        coef = state["coef"]
        return [sum(v * c for v, c in zip(x, coef))]

# The search space is automatically collected from class attributes
print("RidgeRegressor search space:")
for dim in RidgeRegressor._soma_search_space:
    print(f"  {dim}")

# You can use it in a Study by extracting the search space
study = Study(
    name="ridge_tuning",
    search_space=RidgeRegressor._soma_search_space,
    strategy="random",
    n_trials=20,
    objectives=[("loss", "minimize")],
    seed=42,
)

def ridge_objective(params):
    # Simulate: alpha near 1.0 gives best regularization
    alpha = params.get("alpha", 1.0)
    normalize = params.get("normalize", False)
    loss = abs(alpha - 1.0) + (0.1 if not normalize else 0.0)
    return {"loss": loss}

study.run(ridge_objective)
best = study.best_trial
print(f"\nBest: alpha={best['params']['alpha']:.4f}, normalize={best['params']['normalize']}")
print(f"Loss: {best['metrics']['loss']:.4f}")

## 3.7 — Strategy comparison

| Strategy | Best for | Trials | Deterministic |
|---|---|---|---|
| `grid` | Low dimensions (1-3), need exhaustive coverage | `points_per_dim ^ n_dims` | Yes |
| `random` | High dimensions, quick exploration | Exactly `n_trials` | Yes (with seed) |
| `bayesian` | Expensive objectives, need sample efficiency | `n_trials` (startup + TPE) | Yes (with seed) |

---

**Next:** [04 — Streaming](./04_streaming.ipynb) — process data in chunks with different state modes.